In [ ]:
import cv2
from pathlib import Path


In [2]:
FACE_LANDMARKER_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)

def ensure_face_landmarker_model(model_path: str = "face_landmarker.task") -> str:
    path = Path(model_path)
    if path.exists() and path.stat().st_size > 100_000:
        return str(path)
    
    from urllib.request import urlretrieve
    
    print(f"Descargando modelo MediaPipe FaceLandmarker en {path}...")
    urlretrieve(FACE_LANDMARKER_URL, str(path))
    print(f"Modelo descargado ({path.stat().st_size / 1e6:.2f} MB).")
    
    return str(path)

In [3]:
def open_camera_and_options(camera_index: int):
    cap = cv2.VideoCapture(camera_index, cv2.CAP_V4L2)
    if not cap.isOpened():
        print(f"No se pudo abrir la cámara {camera_index}. Verifica la conexión o usa otro índice.")
        return None, None
    
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    return cap

In [4]:
from ironmanhudv2.biometry_engine.engine_config import BiometryConfig, ParamPose3D, ParamEAR, ParamMAR

EAR_PARAMS = ParamEAR(
    threshold=0.21,
    consec_frames_blink=2,
    micro_sleep_seconds=1.2,
    exp_moving_avg_alpha=0.2,
)

MAR_PARAMS = ParamMAR(
    threshold=0.55,
    yawn_seconds=1.0,
    exp_moving_avg_alpha=0.45,
)

POSE3D_PARAMS = ParamPose3D(
    pitch_nod_threshold=-18.0,
    pitch_up_threshold=25.0,
    yaw_look_away_threshold=28.0,
    pose_alert_seconds=1.0,
    exp_moving_avg_alpha=0.4
)

BIOMETRY_CONFIG = BiometryConfig(
    model_path=ensure_face_landmarker_model(),
    params_EAR=EAR_PARAMS,
    params_MAR=MAR_PARAMS,
    params_pose3D=POSE3D_PARAMS,
    alarm_cooldown_seconds=1.8,
    history_len=75
)

In [5]:
from ironmanhudv2.renderer import IronManHUDRenderer

def run_ironman_hud(camera_index = 0, record_gif_path = None, gif_seconds = 30.0):
    cap = open_camera_and_options(camera_index,)
    if cap is None:
        return

    ironman_hud_renderer = IronManHUDRenderer()
    ironman_hud_renderer.setup(BIOMETRY_CONFIG)

    print("Iniciando Ironman HUD. Presiona 'q' para salir.")
    while True:
        ret, frame = cap.read()
        if not ret:
            print("No se pudo leer el cuadro de la cámara.")
            break
        
        hud_frame = ironman_hud_renderer.render(frame)
        cv2.imshow("Ironman HUD", hud_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    
run_ironman_hud()

W0000 00:00:1795554237.024927   12094 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795554237.034241   12098 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795554237.044038   12100 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Iniciando Ironman HUD. Presiona 'q' para salir.


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/skinnydevi/Development/computer-vision/P5/.venv/lib/python3.11/site-packages/cv2/qt/plugins"
